In [11]:
#1.installing spark and setting up
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

print("Spark setup done!!")

Spark setup done!!


In [12]:
# STEP 2: Start Spark
#importing required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, when

# spinning up the spark session locally
spark = SparkSession.builder.appName("MySparkAssignment").getOrCreate()
print("Session started")

Session started


In [13]:
# STEP 3: Load Data
# reading our custom uploaded dataset.csv file into a dataFrame
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

# viewing the column names and data types (schema)
print("Column Names and Data Types (Quick schema)")
df.printSchema()

# viewing the first few rows of the data
print("Displaying First Few Rows ")
df.show()

Column Names and Data Types (Quick schema)
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- region: string (nullable = true)

Displaying First Few Rows 
+---+------+----+---------+------+------+
| id|  name| age| category|salary|region|
+---+------+----+---------+------+------+
|  1|  Alex|  23|     Tech| 50000| North|
|  2|   Ben|  34|       HR| 45000|  East|
|  3| Chris|NULL|Marketing| 38000|  West|
|  4| Diana|  42|     Tech| 70000| North|
|  5|  Evan|  29|       HR| 42000| South|
|  2|   Ben|  34|       HR| 45000|  East|
|  6| Fiona|  31|     Tech|  NULL|  West|
|  7|George|  -5|  Finance| 60000|  East|
|  8|Hannah|  45|Marketing| 52000| North|
+---+------+----+---------+------+------+



In [14]:
# STEP 4: Data Cleaning
# 1. remove duplicate rows from the dataframe
df_no_duplicates = df.dropDuplicates()

# 2. handle missing values (filling missing salaries with the average, dropping null ages)
mean_salary = df_no_duplicates.select(avg("salary")).first()[0]
df_filled = df_no_duplicates.na.fill({"salary": int(mean_salary)})
df_clean_nulls = df_filled.na.drop(subset=["age"])

# 3. check for incorrect or inconsistent data (fixing george's negative age to null)
df_cleaned = df_clean_nulls.withColumn("age", when(col("age") < 0, None).otherwise(col("age")))

print("Cleaned Data Output")
df_cleaned.show()

Cleaned Data Output
+---+------+----+---------+------+------+
| id|  name| age| category|salary|region|
+---+------+----+---------+------+------+
|  2|   Ben|  34|       HR| 45000|  East|
|  5|  Evan|  29|       HR| 42000| South|
|  1|  Alex|  23|     Tech| 50000| North|
|  7|George|NULL|  Finance| 60000|  East|
|  8|Hannah|  45|Marketing| 52000| North|
|  4| Diana|  42|     Tech| 70000| North|
|  6| Fiona|  31|     Tech| 51000|  West|
+---+------+----+---------+------+------+



In [16]:
# STEP 5: Filter Data
# applying simple conditions to filter by age, category, and region
# keeping records where age is 30 or older, and region is not South
df_filtered = df_cleaned.filter((col("age") >= 30) & (col("region") != "South"))

print("Filtered Data Output")
df_filtered.show()

Filtered Data Output
+---+------+---+---------+------+------+
| id|  name|age| category|salary|region|
+---+------+---+---------+------+------+
|  2|   Ben| 34|       HR| 45000|  East|
|  8|Hannah| 45|Marketing| 52000| North|
|  4| Diana| 42|     Tech| 70000| North|
|  6| Fiona| 31|     Tech| 51000|  West|
+---+------+---+---------+------+------+



In [17]:
# STEP 6: Transform Data
# changing data types (making sure age is explicitly cast as an integer)
df_transformed = df_filtered.withColumn("age", col("age").cast("integer"))

# optionally renaming columns if needed for clean layout
df_transformed = df_transformed.withColumnRenamed("name", "employee_name")

print("Transformed Data Schema Check")
df_transformed.printSchema()
df_transformed.show()

Transformed Data Schema Check
root
 |-- id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = false)
 |-- region: string (nullable = true)

+---+-------------+---+---------+------+------+
| id|employee_name|age| category|salary|region|
+---+-------------+---+---------+------+------+
|  2|          Ben| 34|       HR| 45000|  East|
|  8|       Hannah| 45|Marketing| 52000| North|
|  4|        Diana| 42|     Tech| 70000| North|
|  6|        Fiona| 31|     Tech| 51000|  West|
+---+-------------+---+---------+------+------+



In [20]:
# STEP 7: Aggregation
from pyspark.sql.functions import avg, min, max

# performing basic calculations across total rows
print(f"Total rows remaining: {df_transformed.count()}")

# finding average, minimum, and maximum values of the salary column
df_transformed.select(
    avg("salary").alias("average_salary"), min("salary").alias("minimum_salary"),max("salary").alias("maximum_salary")
).show()

Total rows remaining: 4
+--------------+--------------+--------------+
|average_salary|minimum_salary|maximum_salary|
+--------------+--------------+--------------+
|       54500.0|         45000|         70000|
+--------------+--------------+--------------+



In [21]:
# STEP 8: Group Data

from pyspark.sql.functions import count, avg, sum
# using groupBy() to group the filtered data by category
print("--- Grouped Data Summary ---")
df_transformed.groupBy("category").agg(
    count("id").alias("total_count"), avg("salary").alias("average_salary"), sum("salary").alias("total_salary_pool")
).show()

--- Grouped Data Summary ---
+---------+-----------+--------------+-----------------+
| category|total_count|average_salary|total_salary_pool|
+---------+-----------+--------------+-----------------+
|       HR|          1|       45000.0|            45000|
|     Tech|          2|       60500.0|           121000|
|Marketing|          1|       52000.0|            52000|
+---------+-----------+--------------+-----------------+



In [22]:
# STEP 10: Build a Simple Pipeline
# combining all the steps into one final continuous flow and saving it
from pyspark.sql.functions import col, sum

raw_df = spark.read.csv("dataset.csv", header=True, inferSchema=True) # Load

# Clean & Transform steps combined
cleaned_pipeline_df = raw_df.dropDuplicates() \
    .na.fill({"salary": 0}) \
    .withColumn("age", col("age").cast("integer"))

# Filter step
filtered_pipeline_df = cleaned_pipeline_df.filter(col("age") >= 30)

# Aggregate step
final_aggregated_df = filtered_pipeline_df.groupBy("region").agg(sum("salary").alias("total_payroll"))

# Showing the pipeline output
final_aggregated_df.show()

# saving the final aggregated results folder
final_aggregated_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("output_result")
print("Pipeline complete. Output saved.")

+------+-------------+
|region|total_payroll|
+------+-------------+
|  East|        45000|
|  West|            0|
| North|       122000|
+------+-------------+

Pipeline complete. Output saved.


In [23]:
# checking final schema status to confirm validation steps
df_transformed.printSchema()
print("Verification check finished.")

root
 |-- id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = false)
 |-- region: string (nullable = true)

Verification check finished.
